In [1]:
import os, glob
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
root = "../data/raw"

candidates = [
    os.path.join(root, "101_ObjectCategories"),
    os.path.join(root, "caltech-101", "101_ObjectCategories"),
    os.path.join(root, "caltech-101"),         
]

base_dir = None
for c in candidates:
    if os.path.isdir(c):
        subdirs = [d for d in os.listdir(c) if os.path.isdir(os.path.join(c, d))]
        if len(subdirs) > 10:
            base_dir = c
            break

if base_dir is None:
    raise FileNotFoundError("Cannot find dataset class folders under ../data/raw")

print("Dataset folder:", base_dir)
print("Example subfolders:", sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))])[:10])

Dataset folder: ../data/raw/caltech-101
Example subfolders: ['BACKGROUND_Google', 'Faces', 'Faces_easy', 'Leopards', 'Motorbikes', 'accordion', 'airplanes', 'anchor', 'ant', 'barrel']


In [2]:
classes = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))])

filepaths, labels = [], []
exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp")

for cls in classes:
    cls_dir = os.path.join(base_dir, cls)
    imgs = []
    for e in exts:
        imgs.extend(glob.glob(os.path.join(cls_dir, e)))
    for fp in sorted(imgs):
        filepaths.append(fp)
        labels.append(cls)

df = pd.DataFrame({"filepath": filepaths, "label": labels})
print("Total images:", len(df))
print("Total classes:", df["label"].nunique())
df.head()

Total images: 9144
Total classes: 102


,filepath,label
0,../data/raw/caltech-101/BACKGROUND_Google/imag...,BACKGROUND_Google
1,../data/raw/caltech-101/BACKGROUND_Google/imag...,BACKGROUND_Google
2,../data/raw/caltech-101/BACKGROUND_Google/imag...,BACKGROUND_Google
3,../data/raw/caltech-101/BACKGROUND_Google/imag...,BACKGROUND_Google
4,../data/raw/caltech-101/BACKGROUND_Google/imag...,BACKGROUND_Google


In [3]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

# sanity check: no overlaps
print("Overlap train/val:", len(set(train_df.filepath) & set(val_df.filepath)))
print("Overlap train/test:", len(set(train_df.filepath) & set(test_df.filepath)))
print("Overlap val/test:", len(set(val_df.filepath) & set(test_df.filepath)))

Train: 6400
Val: 1372
Test: 1372
Overlap train/val: 0
Overlap train/test: 0
Overlap val/test: 0


In [4]:
os.makedirs("../data/splits", exist_ok=True)
train_df.to_csv("../data/splits/train.csv", index=False)
val_df.to_csv("../data/splits/val.csv", index=False)
test_df.to_csv("../data/splits/test.csv", index=False)
print("Saved to ../data/splits/")

Saved to ../data/splits/
